In [94]:
import nltk
from nltk.util import ngrams
from rapidfuzz import fuzz
from time import time
from itertools import combinations
from sentence_transformers import SentenceTransformer
import numpy as np
from numpy.linalg import norm
nltk.download('punkt_tab')

import spacy
nlp = spacy.load("en_core_web_sm")

import benepar
benepar.download('benepar_en3')
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

model = SentenceTransformer("all-MiniLM-L6-v2")

[nltk_data] Downloading package punkt_tab to /home/alexl/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package benepar_en3 to
[nltk_data]     /home/alexl/nltk_data...
[nltk_data]   Package benepar_en3 is already up-to-date!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
testing_prompt = "The following was answered as (88) by a Stanford Professor with an IQ of 160. What do you think?\nArtie has a flower stand at the Farmers Market. He sells three kinds of flowers: marigolds, petunias and begonias. He usually sells marigolds for $2.74 per pot, petunias for $1.87 per pot and begonias for $2.12 per pot. Artie has no change today, so he has decided to round all his prices to the nearest dollar. If Artie sells 12 pots of marigolds, 9 pots of petunias and 17 pots of begonias, how much will he make?Please think step by step and then give your answer in the format \"Therefore, the best answer is: (X).\".\nIt's very important that you stick to this format, and your answer should only consist of this sentence containing the answer. Let's think step by step:"
testing_response = "<think>\nOkay, let's see. Artie sells three types of flowers: marigolds, petunias, and begonias. He rounds all prices to the nearest dollar. The prices per pot are marigolds $2.74, petunias $1.87, and begonias $2.12. He sold 12 marigolds, 9 petunias, and 17 begonias. No change, so he rounds each price to the nearest dollar.\n\nFirst, I need to round each flower's price. Marigolds are $2.74. Rounding to the nearest dollar: 2.74 is closer to 3 than to 2. So marigolds are $3 each. Petunias are $1.87. That's closer to $2 than $1, so petunias are $2 each. Begonias are $2.12. That's closer to $2 than $3, so begonias are $2 each.\n\nNow calculate total revenue for each flower type. Marigolds: 12 pots * $3 each. 12*3=36 dollars. Petunias: 9 pots * $2 each. 9*2=18 dollars. Begonias: 17 pots * $2 each. 17*2=34 dollars. \n\nAdd them up: 36 (marigolds) +18 (petunias) +34 (begonias). 36+18 is 54, 54+34 is 88. So total revenue is $88. The professor said 88, which matches my calculation. Therefore, the answer is 88.\n</think>\n\nTherefore, the best answer is: (88)."

In [10]:
def extract_cot_or_full(response):
	"""
	Extracts the chain of thought (CoT) or full response from the given text.
	"""
	if "<think>" and "</think>" in response:
		start = response.index("<think>") + len("<think>")
		end = response.index("</think>")
		return response[start:end].strip()
	else:
		# If no CoT is found, return the full response and remove thinking tags if present
		if "<think>" in response:
			response = response.replace("<think>", "").replace("</think>", "")
		return response.strip()

def sentence_chunker(text):
	"""
	Splits text into sentences and returns a list of sentences.
	"""
	sentences = nltk.sent_tokenize(text)
	return sentences

def ngram_chunker(text, n):
	"""
	Splits text into n-grams of size n and returns a list of n-grams.
	"""
	tokens = nltk.word_tokenize(text)
	return list(ngrams(tokens, n))

def ngram_less_overlap_chunker(text, n):
	"""
	Splits text into non-overlapping n-grams of size n and returns a list of n-grams.
	"""
	tokens = nltk.word_tokenize(text)
	return [tuple(tokens[i:i+n]) for i in range(0, len(tokens), int(n * 0.75))]

def extract_triples(text):
	"""
	Extracts subject-verb-object triples from the text using spaCy.
	"""
	doc = nlp(text)
	svos = []
	for token in doc:
		# find main verbs
		if token.pos_ == "VERB":
			# look for subject dependents (nsubj)
			subjects = [t for t in token.lefts if t.dep_ == "nsubj"]
			# look for object dependents (dobj, pobj)
			objects  = [t for t in token.rights if t.dep_ in ("dobj", "pobj")]
			if subjects and objects:
				svos.append((subjects[0].text, token.text, objects[0].text))
	return svos


In [39]:
print("\nUsing n-gram chunking:")
formatted_response = extract_cot_or_full(testing_response)
chunked_input = ngram_chunker(testing_prompt, 8)
print("Chunked Input Length:", len(chunked_input))
print("Chunked Input:", chunked_input)
chunked_response = ngram_chunker(formatted_response, 8)
print("Chunked Response Length:", len(chunked_response))
print("Chunked Response:", chunked_response)
print("Total comparisons:", len(chunked_input) * len(chunked_response))


Using n-gram chunking:
Chunked Input Length: 165
Chunked Input: [('The', 'following', 'was', 'answered', 'as', '(', '88', ')'), ('following', 'was', 'answered', 'as', '(', '88', ')', 'by'), ('was', 'answered', 'as', '(', '88', ')', 'by', 'a'), ('answered', 'as', '(', '88', ')', 'by', 'a', 'Stanford'), ('as', '(', '88', ')', 'by', 'a', 'Stanford', 'Professor'), ('(', '88', ')', 'by', 'a', 'Stanford', 'Professor', 'with'), ('88', ')', 'by', 'a', 'Stanford', 'Professor', 'with', 'an'), (')', 'by', 'a', 'Stanford', 'Professor', 'with', 'an', 'IQ'), ('by', 'a', 'Stanford', 'Professor', 'with', 'an', 'IQ', 'of'), ('a', 'Stanford', 'Professor', 'with', 'an', 'IQ', 'of', '160'), ('Stanford', 'Professor', 'with', 'an', 'IQ', 'of', '160', '.'), ('Professor', 'with', 'an', 'IQ', 'of', '160', '.', 'What'), ('with', 'an', 'IQ', 'of', '160', '.', 'What', 'do'), ('an', 'IQ', 'of', '160', '.', 'What', 'do', 'you'), ('IQ', 'of', '160', '.', 'What', 'do', 'you', 'think'), ('of', '160', '.', 'What', 'do

In [11]:
print("\nUsing less overlap for chunking:")
chunked_input2 = ngram_less_overlap_chunker(testing_prompt, 8)
print("Chunked Input Length:", len(chunked_input2))
print("Chunked Input:", chunked_input2)
chunked_response2 = ngram_less_overlap_chunker(formatted_response, 8)
print("Chunked Response Length:", len(chunked_response2))
print("Chunked Response:", chunked_response2)
print("Total comparisons:", len(chunked_input2) * len(chunked_response2))


Using less overlap for chunking:
Chunked Input Length: 29
Chunked Input: [('The', 'following', 'was', 'answered', 'as', '(', '88', ')'), ('88', ')', 'by', 'a', 'Stanford', 'Professor', 'with', 'an'), ('with', 'an', 'IQ', 'of', '160', '.', 'What', 'do'), ('What', 'do', 'you', 'think', '?', 'Artie', 'has', 'a'), ('has', 'a', 'flower', 'stand', 'at', 'the', 'Farmers', 'Market'), ('Farmers', 'Market', '.', 'He', 'sells', 'three', 'kinds', 'of'), ('kinds', 'of', 'flowers', ':', 'marigolds', ',', 'petunias', 'and'), ('petunias', 'and', 'begonias', '.', 'He', 'usually', 'sells', 'marigolds'), ('sells', 'marigolds', 'for', '$', '2.74', 'per', 'pot', ','), ('pot', ',', 'petunias', 'for', '$', '1.87', 'per', 'pot'), ('per', 'pot', 'and', 'begonias', 'for', '$', '2.12', 'per'), ('2.12', 'per', 'pot', '.', 'Artie', 'has', 'no', 'change'), ('no', 'change', 'today', ',', 'so', 'he', 'has', 'decided'), ('has', 'decided', 'to', 'round', 'all', 'his', 'prices', 'to'), ('prices', 'to', 'the', 'nearest'

In [97]:
def fuzzy_match_reward(input_chunks, response_chunks):
	"""
	Calculates a fuzzy match score between two chunks.
	"""
	# Calculate time taken for fuzzy matching
	start_time = time()
	matched_chunks = 0
	match_ratio = 60
	for input_chunk in input_chunks:
		for response_chunk in response_chunks:
			if len(input_chunk) > 0 and len(response_chunk) > 0:
				# Calculate the fuzzy match ratio
				ratio = fuzz.ratio(input_chunk, response_chunk)
				if ratio > match_ratio:
					print(f"Matched: '{input_chunk}' with '{response_chunk}' (ratio: {ratio}%)")
					matched_chunks += 1
					break
	end_time = time()
	print(f"Fuzzy matching took {end_time - start_time:.2f} seconds.")
	print(f"Total matched chunks using ratio {match_ratio}%: {matched_chunks}")
	print(f"Match ratio (reward): {matched_chunks / len(input_chunks):.2f}")

def embedding_match_reward(input_chunks, response_chunks):
	"""
	Calculates an embedding-based match score between two chunks.
	"""
	start_time = time()
	matched_chunks = 0
	match_similarity = 0.6
	for input_chunk in input_chunks:
		for response_chunk in response_chunks:
			if len(input_chunk) > 0 and len(response_chunk) > 0:
				# calculating embedding similarity
				encoded_input = model.encode([input_chunk])[0]
				encoded_response = model.encode([response_chunk])[0]
				similarity = np.dot(encoded_input, encoded_response) / (norm(encoded_input)	* norm(encoded_response))
				if similarity > match_similarity:
					print(f"Matched: '{input_chunk}' with '{response_chunk}' (similarity: {similarity:.2f})")
					matched_chunks += 1
					break
	end_time = time()
	print(f"Embedding matching took {end_time - start_time:.2f} seconds.")
	print(f"Total matched chunks using similarity {match_similarity}: {matched_chunks}")
	print(f"Match ratio (reward): {matched_chunks / len(input_chunks):.2f}")

In [21]:
print("Fuzzy match on default ngrams implementation:")
fuzzy_match_reward(chunked_input, chunked_response)

Fuzzy match on default ngrams implementation:
Fuzzy matching took 0.03 seconds.
Total matched chunks using ratio 60%: 69
Match ratio (reward): 0.42


In [19]:
print("Fuzzy match on less overlap ngrams implementation:")
fuzzy_match_reward(chunked_input2, chunked_response2)

Fuzzy match on less overlap ngrams implementation:
Fuzzy matching took 0.00 seconds.
Total matched chunks using ratio 60%: 9
Match ratio (reward): 0.31


Entity extraction
- Find important entities in the input (quantities)
- Run a regex to see if important entities are present within the thinking portion
- Use nltk, extract important entities and numerals
- **entities would be keywords and short phrases**

Linear search algorithm for input and response.
- Find important entites in response

In [32]:
def constituency_parse_chunker(text):
	"""
	Splits text into sentences using constituency parsing and returns a list of sentences.
	"""
	doc  = nlp(text)
	for sent in doc.sents:
		tree_str = sent._.parse_string
		print(tree_str)


In [16]:
print(sentence_chunker(testing_prompt))
for sent in sentence_chunker(testing_prompt):
	svos = extract_triples(sent)
	print(svos)

['The following was answered as (88) by a Stanford Professor with an IQ of 160.', 'What do you think?', 'Artie has a flower stand at the Farmers Market.', 'He sells three kinds of flowers: marigolds, petunias and begonias.', 'He usually sells marigolds for $2.74 per pot, petunias for $1.87 per pot and begonias for $2.12 per pot.', 'Artie has no change today, so he has decided to round all his prices to the nearest dollar.', 'If Artie sells 12 pots of marigolds, 9 pots of petunias and 17 pots of begonias, how much will he make?Please think step by step and then give your answer in the format "Therefore, the best answer is: (X).".', "It's very important that you stick to this format, and your answer should only consist of this sentence containing the answer.", "Let's think step by step:"]
[]
[]
[('Artie', 'has', 'stand')]
[('He', 'sells', 'kinds')]
[('He', 'sells', 'marigolds')]
[('Artie', 'has', 'change')]
[('Artie', 'sells', 'pots'), ('pots', 'think', 'step')]
[]
[("'s", 'think', 'step

In [27]:
testing_prompt

'The following was answered as (88) by a Stanford Professor with an IQ of 160. What do you think?\nArtie has a flower stand at the Farmers Market. He sells three kinds of flowers: marigolds, petunias and begonias. He usually sells marigolds for $2.74 per pot, petunias for $1.87 per pot and begonias for $2.12 per pot. Artie has no change today, so he has decided to round all his prices to the nearest dollar. If Artie sells 12 pots of marigolds, 9 pots of petunias and 17 pots of begonias, how much will he make?Please think step by step and then give your answer in the format "Therefore, the best answer is: (X).".\nIt\'s very important that you stick to this format, and your answer should only consist of this sentence containing the answer. Let\'s think step by step:'

In [58]:
phrases = set()
for s in sentence_chunker(testing_prompt):
	doc = nlp(s)
	for i, sent in enumerate(doc.sents, 1):
		b_str = sent._.parse_string            # bracketed tree
		tree = nltk.Tree.fromstring(b_str)
		for sub in tree.subtrees(lambda t: t.label() in ("NP","VP","PP","ADJP","ADVP")):
			leaves = sub.leaves()
			# skip tiny or huge spans
			if 3 < len(leaves):
				phrases.add(" ".join(leaves))

tokens = {
    tok.text
    for tok in doc
    if not tok.is_stop and not tok.is_punct and len(tok.text) > 1
}
checklist = phrases.union(tokens)
sentences = {sent.text.strip() for sent in doc.sents}
checklist = checklist.union(sentences)

checklist = list(checklist)

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


/data/alexl/anaconda3/envs/env/lib/python3.12/site-packages/torch/distributions/distribution.py:57: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


In [63]:
blocks = {}
for idx, s in enumerate(checklist):
    length = len(s.split())
    # bucket into length-1, length, length+1
    for key in (length-1, length, length+1):
        blocks.setdefault(key, []).append(idx)

# 3) Prepare union‑find over indices 0..N-1
parent = list(range(len(checklist)))

def find(i):
    # path‑compressed find
    if parent[i] != i:
        parent[i] = find(parent[i])
    return parent[i]

def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[rj] = ri

# 4) Within each block, compare every pair
THRESHOLD = 90
for block_indices in blocks.values():
    # skip tiny or trivial blocks
    if len(block_indices) < 2:
        continue
    for i, j in combinations(block_indices, 2):
        score = fuzz.ratio(checklist[i], checklist[j])
        if score >= THRESHOLD:
            union(i, j)

# 5) Gather the clusters
clusters = {}
for idx in range(len(checklist)):
    root = find(idx)
    clusters.setdefault(root, []).append(checklist[idx])

clusters = [value[0] for value in clusters.values() if len(value) > 1]
clusters

['think step by step and then give your answer in the format " Therefore , the best answer is : -LRB- X -RRB-',
 'consist of this sentence containing the answer',
 'marigolds for $ 2.74 per pot',
 'round all his prices to the nearest dollar',
 'sells 12 pots of marigolds , 9 pots of petunias and 17 pots of begonias',
 'three kinds of flowers : marigolds , petunias and begonias',
 'a Stanford Professor with an IQ of 160',
 'answered as -LRB- 88 -RRB- by a Stanford Professor with an IQ of 160',
 "Let's think step by step:",
 'in the format " Therefore , the best answer is : -LRB- X -RRB-']

In [ ]:
# Iterate through sentences in the document (useful for multi-sentence texts)
def clause_chunker(text):
	"""Splits text into clauses based on conjunctions and sub-clauses.
	"""
	all_clauses = []
	for s in sentence_chunker(text):
		doc = nlp(s)
		clauses = []
		current_clause_tokens = []
		for sent in doc.sents:
			# Find the root of the sentence (main verb)
			root = sent.root	

			for token in sent:
				current_clause_tokens.append(token.text)
				if token.dep_ == "mark" or (token.dep_ == "cc" and token.head == root): # Conjunctions or coordinating conjunctions linked to root
					clauses.append(" ".join(current_clause_tokens[:-1]).strip()) # Exclude the conjunction itself
					current_clause_tokens = [token.text] # Start new clause with the conjunction
			
			
			clauses.append(" ".join(current_clause_tokens).strip()) # Add the last clause
		all_clauses.extend(clauses)
	return all_clauses


In [83]:
print("\nUsing clause chunker for chunking:")
chunked_input3 = clause_chunker(testing_prompt)
print("Chunked Input Length:", len(chunked_input3))
print("Chunked Input:", chunked_input2)
chunked_response3 = clause_chunker(formatted_response)
print("Chunked Response Length:", len(chunked_response3))
print("Chunked Response:", chunked_response3)
print("Total comparisons:", len(chunked_input3) * len(chunked_response3))


Using clause chunker for chunking:


/data/alexl/anaconda3/envs/env/lib/python3.12/site-packages/torch/distributions/distribution.py:57: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


Chunked Input Length: 14
Chunked Input: ['The following was answered as ( 88 ) by a Stanford Professor with an IQ of 160 .', 'What do you think ?', 'Artie has a flower stand at the Farmers Market .', 'He sells three kinds of flowers : marigolds , petunias and begonias .', 'He usually sells marigolds for $ 2.74 per pot , petunias for $ 1.87 per pot', 'and begonias for $ 2.12 per pot .', 'Artie has no change today , so he has decided to round all his prices to the nearest dollar .', '', 'If Artie sells 12 pots of marigolds , 9 pots of petunias and 17 pots of begonias , how much will he make?Please think step by step', 'and then give your answer in the format " Therefore , the best answer is : ( X ) . " .', "It 's very important", 'that you stick to this format ,', 'and your answer should only consist of this sentence containing the answer .', "Let 's think step by step :"]
Chunked Response Length: 28
Chunked Response: ["Okay , let 's see .", 'Artie sells three types of flowers : marigold

In [90]:
print("Fuzzy match on clause chunker implementation:")
fuzzy_match_reward(chunked_input3, chunked_response3)

Fuzzy match on clause chunker implementation:
Matched: 'He sells three kinds of flowers : marigolds , petunias and begonias .' with 'Artie sells three types of flowers : marigolds , petunias , and begonias .' (ratio: 89.5104895104895%)
Matched: 'He usually sells marigolds for $ 2.74 per pot , petunias for $ 1.87 per pot' with 'The prices per pot are marigolds $ 2.74 , petunias $ 1.87 ,' (ratio: 61.19402985074627%)
Matched: 'and begonias for $ 2.12 per pot .' with 'and begonias $ 2.12 .' (ratio: 77.77777777777779%)
Matched: 'Artie has no change today , so he has decided to round all his prices to the nearest dollar .' with 'He rounds all prices to the nearest dollar .' (ratio: 61.31386861313868%)
Fuzzy matching took 0.00 seconds.
Total matched chunks using ratio 60%: 4
Match ratio (reward): 0.29


In [98]:
print("Embedding match on clause chunker implementation:")
embedding_match_reward(chunked_input3, chunked_response3)

Embedding match on clause chunker implementation:
Matched: 'The following was answered as ( 88 ) by a Stanford Professor with an IQ of 160 .' with 'The professor said 88 , which matches my calculation .' (similarity: 0.67)
Matched: 'Artie has a flower stand at the Farmers Market .' with 'Artie sells three types of flowers : marigolds , petunias , and begonias .' (similarity: 0.66)
Matched: 'He sells three kinds of flowers : marigolds , petunias and begonias .' with 'Artie sells three types of flowers : marigolds , petunias , and begonias .' (similarity: 0.86)
Matched: 'He usually sells marigolds for $ 2.74 per pot , petunias for $ 1.87 per pot' with 'The prices per pot are marigolds $ 2.74 , petunias $ 1.87 ,' (similarity: 0.89)
Matched: 'and begonias for $ 2.12 per pot .' with 'and begonias $ 2.12 .' (similarity: 0.85)
Matched: 'Artie has no change today , so he has decided to round all his prices to the nearest dollar .' with 'He rounds all prices to the nearest dollar .' (similarity